# Python drills 6: files, paths and formats

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-drills-06-files-and-formats.ipynb)

Companion to the [Files, paths and formats](../learning/python-06-files-and-formats.ipynb) chapter. The exercises read and write CSV, JSON, JSON Lines and text. They use a temporary folder, which the last cell deletes.

<details>
<summary><b>How to use this notebook</b></summary>

Work through the exercises in order. Each one has three parts:

1. **Question:** what to build, with the rules it must follow.
2. **Exercise cell:** replace the `pass` with your code. Keep the function name and parameters.
3. **Check cell:** run it after your code. Each line says `pass` when your function gives the expected result, and `todo` when it does not yet. Checks never stop the notebook, so you can run everything at any time.

The last cell prints your score. Use `Runtime -> Run all` to rerun everything after a change.

</details>

In [ ]:
_tally = {"passed": 0, "total": 0}

def check(label, make_actual, expected):
    """Run one check. It never raises, so the notebook always runs from top to bottom."""
    _tally["total"] += 1
    try:
        actual = make_actual()
    except Exception as error:
        print(f"error  {label}: {type(error).__name__}: {error}")
        return
    if actual == expected:
        _tally["passed"] += 1
        print(f"pass   {label}")
    else:
        print(f"todo   {label}: got {actual!r}")

def raises(make_actual, error_type):
    """True when the call raises error_type, False when it raises anything else or nothing."""
    try:
        make_actual()
    except error_type:
        return True
    except Exception:
        return False
    return False

def score():
    print(f"\nScore: {_tally['passed']} of {_tally['total']} checks pass")

In [ ]:
import tempfile
from pathlib import Path

WORKDIR = Path(tempfile.mkdtemp(prefix="practice-files-"))

def scratch(name, text):
    """Write `text` to a file in the temporary folder and return its path."""
    path = WORKDIR / name
    path.write_text(text, encoding="utf-8")
    return path

## Exercise 1: count the non-blank lines

**Difficulty:** Medium · about 10 min

Write `count_nonblank_lines(path)`. Count the lines that contain something other than spaces. Read the file line by line, so a large file never has to fit in memory.

In [ ]:
def count_nonblank_lines(path):
    # your code here
    pass

In [ ]:
check(
    "count_nonblank_lines skips blank and space-only lines",
    lambda: count_nonblank_lines(scratch("lines.txt", "one\n\n  \ntwo\n")),
    2,
)
check('empty file', lambda: count_nonblank_lines(scratch('empty.txt', '')), 0)
check('spaces only', lambda: count_nonblank_lines(scratch('spaces.txt', '  \n\t\n')), 0)
check('no trailing newline', lambda: count_nonblank_lines(scratch('last.txt', 'last')), 1)
check('unicode', lambda: count_nonblank_lines(scratch('unicode.txt', 'café\n東京\n')), 2)
check('many lines', lambda: count_nonblank_lines(scratch('many.txt', 'a\n' * 1000)), 1000)

<details>
<summary><b>Show solution</b></summary>

```python
def count_nonblank_lines(path):
    with path.open(encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())
```

</details>

## Exercise 2: read a CSV file with typed values

**Difficulty:** Medium · about 10 min

Write `read_orders(path)`. Read the CSV file with `csv.DictReader`. Return a list of dictionaries where `order_id` is an `int` and `amount` is a `float`. The CSV columns are `order_id` and `amount`.

In [ ]:
import csv

def read_orders(path):
    # your code here
    pass

In [ ]:
check(
    "read_orders converts the types",
    lambda: read_orders(scratch("orders.csv", "order_id,amount\n1,9.5\n2,10\n")),
    [{"order_id": 1, "amount": 9.5}, {"order_id": 2, "amount": 10.0}],
)
check('header only', lambda: read_orders(scratch('header.csv', 'order_id,amount\n')), [])
check('zero amount', lambda: read_orders(scratch('zero.csv', 'order_id,amount\n0,0\n')), [{'order_id': 0, 'amount': 0.0}])
check('negative amount', lambda: read_orders(scratch('negative.csv', 'order_id,amount\n3,-2.5\n')), [{'order_id': 3, 'amount': -2.5}])
check('quoted fields', lambda: read_orders(scratch('quoted.csv', 'order_id,amount\n"4","1.25"\n')), [{'order_id': 4, 'amount': 1.25}])
check('invalid id rejected', lambda: raises(lambda: read_orders(scratch('bad.csv', 'order_id,amount\nx,2\n')), ValueError), True)

<details>
<summary><b>Show solution</b></summary>

```python
import csv

def read_orders(path):
    with path.open(newline="", encoding="utf-8") as handle:
        return [
            {"order_id": int(row["order_id"]), "amount": float(row["amount"])}
            for row in csv.DictReader(handle)
        ]
```

</details>

## Exercise 3: write JSON Lines with sorted keys

**Difficulty:** Medium · about 10 min

Write `write_jsonl(path, records)`. Write one JSON object per line, with the keys in sorted order. Return the number of records written.

*Hint: `json.dumps(record, sort_keys=True)` gives one object's text.*

In [ ]:
import json

def write_jsonl(path, records):
    # your code here
    pass

In [ ]:
check(
    "write_jsonl returns the count",
    lambda: write_jsonl(WORKDIR / "count.jsonl", [{"b": 1}, {"c": 3}]),
    2,
)
check(
    "write_jsonl writes sorted keys, one per line",
    lambda: (lambda p: (write_jsonl(p, [{"b": 1, "a": 2}, {"c": 3}]), p.read_text(encoding="utf-8"))[1])(WORKDIR / "out.jsonl"),
    '{"a": 2, "b": 1}\n{"c": 3}\n',
)
check('empty iterator', lambda: write_jsonl(WORKDIR / 'empty.jsonl', iter([])), 0)
check('empty output', lambda: (write_jsonl(WORKDIR / 'empty2.jsonl', []), (WORKDIR / 'empty2.jsonl').read_text())[1], '')
check('generator', lambda: write_jsonl(WORKDIR / 'gen.jsonl', ({'id': i} for i in range(3))), 3)
check('one empty object', lambda: (write_jsonl(WORKDIR / 'obj.jsonl', [{}]), (WORKDIR / 'obj.jsonl').read_text())[1], '{}\n')

<details>
<summary><b>Show solution</b></summary>

```python
import json

def write_jsonl(path, records):
    count = 0
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, sort_keys=True) + "\n")
            count += 1
    return count
```

</details>

## Exercise 4: dump a record that has dates and decimals

**Difficulty:** Medium · about 10 min

Write `dump_record(record)`. Return a JSON string with sorted keys. Convert each `date` to its ISO text, and each `Decimal` to a string, so the exact digits are kept. Other values are left as they are.

*Hint: `json.dumps` accepts a `default=` function that is called for values it cannot encode.*

In [ ]:
import json
from datetime import date
from decimal import Decimal

def dump_record(record):
    # your code here
    pass

In [ ]:
check(
    "dump_record handles date and Decimal",
    lambda: dump_record({"day": date(2024, 1, 15), "total": Decimal("12.50"), "id": 7}),
    '{"day": "2024-01-15", "id": 7, "total": "12.50"}',
)
check('empty record', lambda: dump_record({}), '{}')
check('plain values', lambda: dump_record({'x': None, 'ok': True}), '{"ok": true, "x": null}')
check('exact decimal', lambda: dump_record({'v': Decimal('0.1000')}), '{"v": "0.1000"}')
check('nested date', lambda: dump_record({'v': [date(2024, 2, 29)]}), '{"v": ["2024-02-29"]}')
check('unsupported type', lambda: raises(lambda: dump_record({'v': {1, 2}}), TypeError), True)

<details>
<summary><b>Show solution</b></summary>

```python
import json
from datetime import date
from decimal import Decimal

def dump_record(record):
    def encode(value):
        if isinstance(value, date):
            return value.isoformat()
        if isinstance(value, Decimal):
            return str(value)
        raise TypeError(f"cannot encode {type(value).__name__}")
    return json.dumps(record, default=encode, sort_keys=True)
```

</details>

## Exercise 5: read the first line without the byte-order mark

**Difficulty:** Medium · about 10 min

Write `first_line_without_bom(path)`. Open the file so that a leading byte-order mark is removed, then return the first line without its line ending.

*Hint: the `utf-8-sig` encoding strips the mark.*

In [ ]:
def first_line_without_bom(path):
    # your code here
    pass

In [ ]:
check(
    "first_line_without_bom strips the BOM",
    lambda: first_line_without_bom(scratch("bom.csv", "﻿id,name\n1,ana\n")),
    "id,name",
)
check('empty file', lambda: first_line_without_bom(scratch('emptybom.txt', '')), '')
check('plain text', lambda: first_line_without_bom(scratch('plain.txt', 'abc\ndef')), 'abc')
check('no newline', lambda: first_line_without_bom(scratch('nonewline.txt', 'abc')), 'abc')
check('keep spaces', lambda: first_line_without_bom(scratch('spacebom.txt', '  abc  \n')), '  abc  ')
check('only BOM', lambda: first_line_without_bom(scratch('onlybom.txt', '\ufeff')), '')

<details>
<summary><b>Show solution</b></summary>

```python
def first_line_without_bom(path):
    with path.open(encoding="utf-8-sig") as handle:
        return handle.readline().rstrip("\n")
```

</details>

## Exercise 6: write a text file atomically

**Difficulty:** Medium · about 10 min

Write `write_text_atomic(path, text)`. Write the text to a temporary file named `<name>.tmp` in the same folder, then `replace` it onto `path`. Return `path`. No `.tmp` file may remain afterwards. On write or replacement failure, keep the original content, remove the temporary file, and re-raise the error.

In [ ]:
def write_text_atomic(path, text):
    # your code here
    pass

In [ ]:
from unittest.mock import patch
import os

def atomic_failure_case(where):
    path = scratch('preserve.txt', 'original')
    tmp = path.with_name(path.name + '.tmp')
    try:
        if where == 'replace':
            with patch.object(Path, 'replace', side_effect=OSError('replace failed')):
                write_text_atomic(path, 'new')
        else:
            original_write = Path.write_text
            def fail_write(target, text, *args, **kwargs):
                if target == tmp:
                    original_write(target, 'partial', *args, **kwargs)
                    raise OSError('write failed')
                return original_write(target, text, *args, **kwargs)
            with patch.object(Path, 'write_text', fail_write):
                write_text_atomic(path, 'new')
    except OSError:
        return path.read_text() == 'original' and not tmp.exists()
    return False

check(
    "write_text_atomic writes the text and leaves no .tmp file",
    lambda: (lambda p: (write_text_atomic(p, "hi"), p.read_text(encoding="utf-8"), p.with_name(p.name + ".tmp").exists()))(WORKDIR / "atomic.txt")[1:],
    ("hi", False),
)
check('returns path', lambda: write_text_atomic(WORKDIR / 'return.txt', 'a') == WORKDIR / 'return.txt', True)
check('empty text', lambda: (write_text_atomic(WORKDIR / 'blank.txt', ''), (WORKDIR / 'blank.txt').read_text())[1], '')
check('unicode text', lambda: (write_text_atomic(WORKDIR / 'utf.txt', 'café'), (WORKDIR / 'utf.txt').read_text(encoding='utf-8'))[1], 'café')
check('replacement preserves old file on error', lambda: atomic_failure_case('replace'), True)
check('write failure preserves old file', lambda: atomic_failure_case('write'), True)

<details>
<summary><b>Show solution</b></summary>

```python
def write_text_atomic(path, text):
    tmp = path.with_name(path.name + ".tmp")
    try:
        tmp.write_text(text, encoding="utf-8")
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)
    return path
```

</details>

## Score

In [ ]:
score()

## Clean up

Delete the temporary folder. Nothing after this cell uses it.

In [ ]:
import shutil

shutil.rmtree(WORKDIR)
assert not WORKDIR.exists()